<a href="https://colab.research.google.com/github/liyenrondon/IA-2/blob/main/resumen_video_orden_del_caos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# APRENDIZAJE NO SUPERVISADO: CLUSTERING, ANOMALÍAS Y GMM
# ==============================================================================

# ------------------------------------------------------------------------------
# 0. Importación de librerías necesarias
# ------------------------------------------------------------------------------
import numpy as np                                       # Librería para operaciones numéricas y matrices
import matplotlib.pyplot as plt                          # Librería para graficar resultados
from sklearn.datasets import make_blobs, make_moons      # Generadores de conjuntos de datos sintéticos
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN # Algoritmos de clustering
from sklearn.metrics import silhouette_score, silhouette_samples # Métricas para evaluar clústeres
from sklearn.mixture import GaussianMixture              # Modelo de Mezclas Gaussianas (GMM)


# ==============================================================================
# 1. ALGORITMOS DE AGRUPAMIENTO (CLUSTERING)
# ==============================================================================

# --- Generación de datos sintéticos sencillos ---
# Generamos 300 puntos distribuidos en 4 centros distintos para probar k-means
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=42)

# --- k-means y k-means++ ---
# Inicializamos el modelo K-Means especificando 4 clústeres e inicialización inteligente 'k-means++'
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
# Ajustamos el modelo con nuestros datos y obtenemos las etiquetas asignadas a cada punto
labels_kmeans = kmeans.fit_predict(X)
# Obtenemos las coordenadas de los centroides finales aprendidos por el algoritmo
centroids = kmeans.cluster_centers_

# Graficamos el resultado de K-Means
plt.figure(figsize=(6, 4))                               # Definimos el tamaño de la figura
plt.scatter(X[:, 0], X[:, 1], c=labels_kmeans, s=30, cmap='viridis') # Pintamos los puntos según su clúster
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, label='Centroides') # Dibujamos los centroides
plt.title("Clustering con K-Means (Teselación de Voronoi)") # Título de la gráfica
plt.legend()                                             # Mostramos la leyenda
plt.show()                                               # Desplegamos la imagen

# --- Mini-batch k-means ---
# MiniBatchKMeans procesa subconjuntos pequeños (batches) para acelerar el entrenamiento en grandes volúmenes de datos
minibatch_kmeans = MiniBatchKMeans(n_clusters=4, batch_size=32, random_state=42, n_init=10)
minibatch_kmeans.fit(X)                                  # Entrenamos el modelo rápido

# --- Selección del número de clústeres (Inercia / Método del Codo) ---
inertias = []                                            # Lista vacía para almacenar la inercia de cada 'k'
k_range = range(1, 10)                                   # Probaremos desde k=1 hasta k=9
for k in k_range:                                        # Iteramos sobre cada número de clústeres
    km = KMeans(n_clusters=k, random_state=42, n_init=10) # Creamos el modelo para 'k'
    km.fit(X)                                            # Entrenamos el modelo
    inertias.append(km.inertia_)                         # Guardamos la inercia (suma de distancias cuadradas)

# Graficamos el método del codo
plt.figure(figsize=(6, 4))                               # Definimos tamaño de gráfica
plt.plot(k_range, inertias, 'bo-')                       # Graficamos puntos azúles unidos por línea
plt.xlabel("Número de Clústeres (k)")                   # Nombre del eje X
plt.ylabel("Inercia")                                    # Nombre del eje Y
plt.title("Método del Codo para Selección de k")        # Título
plt.show()                                               # Desplegamos la imagen

# --- Coeficiente de Silueta ---
# Calculamos la silueta promedio para k=4 (mide qué tan bien separados y homogéneos están los grupos)
sil_score = silhouette_score(X, labels_kmeans)
print(f"Coeficiente de Silueta Promedio para k=4: {sil_score:.3f}")

# --- DBSCAN (Agrupamiento basado en densidad) ---
# Generamos datos en forma de medias lunas, complejos de agrupar para K-Means
X_moons, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
# Instanciamos DBSCAN definiendo el radio de búsqueda (eps) y el mínimo de muestras vecinas
dbscan = DBSCAN(eps=0.2, min_samples=5)
# Ajustamos y predecimos las etiquetas (-1 indica ruido / atípico)
labels_dbscan = dbscan.fit_predict(X_moons)

# Graficamos DBSCAN
plt.figure(figsize=(6, 4))                               # Tamaño de la gráfica
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_dbscan, cmap='plasma') # Pintamos los grupos y el ruido
plt.title("DBSCAN: Formas Complejas e Identificación de Ruido") # Título
plt.show()                                               # Desplegamos la imagen


# ==============================================================================
# APLICACIONES PRÁCTICAS DEL CLUSTERING
# ==============================================================================

# --- Segmentación de imágenes (Simplificación por color) ---
# Generamos un lienzo sintético con colores para simular una imagen (100x100 píxeles, 3 canales RGB)
img_dummy = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
# Reestructuramos la imagen a una lista plana de píxeles (10000 píxeles x 3 colores)
X_img = img_dummy.reshape(-1, 3)
# Usamos K-Means para reducir la paleta completa a solo 4 colores principales
kmeans_img = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_img)
# Reemplazamos cada color original por el color del centroide correspondiente
segmented_img = kmeans_img.cluster_centers_[kmeans_img.labels_].reshape(100, 100, 3).astype(np.uint8)

# --- Aprendizaje semisupervisado (Propagación de etiquetas) ---
# 1. Agrupamos los datos en 4 clústeres mediante K-Means
# 2. Encontramos la instancia más cercana a cada centroide para etiquetarla manualmente
representative_digit_idx = np.argmin(kmeans.transform(X), axis=0) # Índices representativos
# 3. Propagamos la etiqueta elegida a todos los miembros de ese mismo clúster

# --- Reducción de dimensionalidad y extracción de características ---
# Convertimos cada punto original a un vector con la distancia a cada uno de los 4 centroides
X_transformed = kmeans.transform(X)                      # Ahora cada instancia tiene 4 dimensiones


# ==============================================================================
# 2. DETECCIÓN DE ANOMALÍAS Y NOVEDADES
# ==============================================================================

# --- Detección de Anomalías (Datos con ruido previo) ---
# Entrenamos el modelo con datos que contienen observaciones atípicas en regiones de baja densidad
dbscan_anomaly = DBSCAN(eps=0.3, min_samples=5).fit(X)
# Las observaciones marcadas con -1 son identificadas como anomalías
anomalies = X[dbscan_anomaly.labels_ == -1]

# --- Detección de Novedades ---
# A diferencia de las anomalías, el conjunto de entrenamiento original está 100% limpio de atípicos.
# Si llega una nueva observación durante producción, se compara con el modelo limpio para decidir si es novedad.


# ==============================================================================
# 3. MODELOS DE MEZCLAS GAUSSIANAS (GMM)
# ==============================================================================

# --- Modelado Probabilístico y Algoritmo Esperanza-Maximización (EM) ---
# Inicializamos el GMM definiendo 4 componentes gaussianas
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
# El algoritmo EM ajusta la media, covarianza (forma/orientación) y pesos de cada gaussiana
gmm.fit(X)

# Asignamos probabilidades (Soft Clustering) a cada instancia en lugar de una etiqueta rígida
probabilities = gmm.predict_proba(X)                     # Retorna una matriz con las probabilidades para cada grupo

# --- Estimación de densidad y detección de atípicos ---
# score_samples calcula el logaritmo de la función de densidad de probabilidad (log-likelihood)
densities = gmm.score_samples(X)
# Calculamos un umbral basado en el 5% de las densidades más bajas
density_threshold = np.percentile(densities, 5)
# Identificamos los puntos cuya densidad está por debajo del umbral como anomalías
gmm_anomalies = X[densities < density_threshold]

# Graficamos los atípicos detectados por GMM
plt.figure(figsize=(6, 4))                               # Tamaño de gráfica
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, label='Puntos Normales') # Pintamos todos los puntos
plt.scatter(gmm_anomalies[:, 0], gmm_anomalies[:, 1], c='red', label='Anomalías (Baja densidad)') # Resaltamos atípicos
plt.title("Detección de Anomalías usando GMM (score_samples)") # Título
plt.legend()                                             # Mostrar leyenda
plt.show()                                               # Desplegar gráfica

# --- Selección del número de componentes mediante AIC y BIC ---
bics = []                                                # Lista para almacenar valores de BIC
aics = []                                                # Lista para almacenar valores de AIC
n_components_range = range(1, 7)                         # Evaluamos de 1 a 6 componentes

for n in n_components_range:                             # Iteramos en el rango definido
    gmm_eval = GaussianMixture(n_components=n, random_state=42).fit(X) # Entrenamos GMM con 'n' componentes
    bics.append(gmm_eval.bic(X))                         # Guardamos el Criterio de Información Bayesiano
    aics.append(gmm_eval.aic(X))                         # Guardamos el Criterio de Información de Akaike

# Graficamos las curvas BIC y AIC (los valores más bajos indican la cantidad óptima de componentes)
plt.figure(figsize=(6, 4))                               # Tamaño de gráfica
plt.plot(n_components_range, bics, label='BIC', marker='o') # Curva BIC
plt.plot(n_components_range, aics, label='AIC', marker='s') # Curva AIC
plt.xlabel("Número de Componentes")                     # Etiqueta eje X
plt.ylabel("Puntuación de Criterio")                    # Etiqueta eje Y
plt.title("Selección de Componentes en GMM (AIC vs BIC)") # Título
plt.legend()                                             # Mostrar leyenda
plt.show()                                               # Desplegar gráfica